## 引导 + 配置

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "cv" else Path.cwd()
sys.path.insert(0, str(ROOT))

from common.config import load_config, with_overrides
from common.utils import set_seed, get_device, ensure_dir

cfg = load_config("config.yaml")
cfg = with_overrides(cfg, epochs=3)
cfg

## 数据

In [ ]:
from torchvision import transforms
from torchvision.datasets import FakeData
import torch
from torch.utils.data import DataLoader

def make_loaders(cfg):
    tf = transforms.Compose([
        transforms.Resize((int(cfg["image_size"]), int(cfg["image_size"]))),
        transforms.ToTensor()
    ])
    ds = FakeData(
        size=int(cfg["n_samples"]),
        image_size=(3, int(cfg["image_size"]), int(cfg["image_size"])),
        num_classes=int(cfg["num_classes"]),
        transform=tf,
        random_offset=int(cfg["seed"])
    )
    n_val = max(1, int(len(ds) * 0.2))
    n_tr = len(ds) - n_val
    tr, va = torch.utils.data.random_split(ds, [n_tr, n_val], generator=torch.Generator().manual_seed(int(cfg["seed"])))
    bs, nw = int(cfg["batch_size"]), int(cfg["num_workers"])
    return (DataLoader(tr, batch_size=bs, shuffle=True, num_workers=nw),
            DataLoader(va, batch_size=bs, shuffle=False, num_workers=nw))

## 模型

In [ ]:
import torch.nn as nn
from torchvision import models

def build_model(cfg) -> nn.Module:
    weights = models.ResNet18_Weights.DEFAULT if cfg["pretrained"] else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, int(cfg["num_classes"]))
    return model

## 训练主流程

In [ ]:
from common.checkpoint import init_early_stopping
from common.trainer import fit

def run_training(cfg, out_dir):
    device = get_device(cfg["device"])
    set_seed(int(cfg["seed"]))
    train_loader, val_loader = make_loaders(cfg)
    model = build_model(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["weight_decay"]))
    criterion = nn.CrossEntropyLoss()
    es_state = init_early_stopping(int(cfg["patience"]), out_dir / "best.pt")
    model, history, es_state = fit(
        model, train_loader, val_loader, optimizer, criterion, device,
        epochs=int(cfg["epochs"]), es_state=es_state
    )
    return model, history, es_state

out_dir = ensure_dir(cfg["output_dir"])
model, history, es_state = run_training(cfg, out_dir)
print("best epoch:", es_state["best_epoch"], "| best val_loss:", round(es_state["best"], 4))

## 画图

In [ ]:
from common.plot import plot_history

plot_history(history, out_dir / "curves.png")

## 收集预测 + 混淆矩阵

In [ ]:
from common.checkpoint import load_checkpoint
from common.plot import plot_confusion_matrix

@torch.no_grad()
def collect_preds(model, loader, device):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        ps.extend(logits.argmax(1).cpu().tolist())
        ys.extend(yb.tolist())
    return ys, ps

device = get_device(cfg["device"])
_, val_loader = make_loaders(cfg)
best = load_checkpoint(build_model(cfg), out_dir / "best.pt", device).to(device)
ys, ps = collect_preds(best, val_loader, device)
plot_confusion_matrix(ys, ps, out_dir / "confusion.png")
print("val acc:", sum(y == p for y, p in zip(ys, ps)) / len(ys))

## 样本图网络

In [ ]:
import matplotlib.pyplot as plt

def show_batch(loader, n=8):
    xb, yb = next(iter(loader))
    n = min(n, xb.size(0))
    fig, axes = plt.subplots(1, n, figsize=(n * 1.6, 2))
    for i in range(n):
        img = xb[i].permute(1, 2, 0).numpy()
        axes[i].imshow(img)
        axes[i].set_title(str(yb[i].item()))
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

show_batch(val_loader, n=8)

## 推理闭环

In [ ]:
from PIL import Image

def predict_one(model, image, cfg, device):
    tf = transforms.Compose([
        transforms.Resize((int(cfg["image_size"]), int(cfg["image_size"]))),
        transforms.ToTensor()
    ])
    x = tf(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        prob = torch.softmax(logits, dim=1)
    pred = prob.argmax(1).item()
    return pred, prob[0, pred].item()

xb, yb = next(iter(val_loader))
img = transforms.ToPILImage()(xb[0])
pred, conf = predict_one(best, img, cfg, device)
print(f"true={yb[0].item()} pred={pred} conf={conf:.3f}")